# eval.ipynb — Grid Outage Forecaster: Held-Out Evaluation
**Challenge:** T2.3 · AIMS KTT Hackathon

This notebook evaluates the trained model on the last 30 days of held-out data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from src.forecaster import OutageForecaster, create_features, FEATURE_COLS
from src.prioritizer import AppliancePrioritizer
from src.evaluate import evaluate_model, print_report

BASE_DIR = Path('..')
MODEL_PATH = BASE_DIR / 'models' / 'outage_model.pkl'
CSV_PATH = BASE_DIR / 'data' / 'raw' / 'grid_history.csv'

print('Paths OK:', MODEL_PATH.exists(), CSV_PATH.exists())

## 1. Load Data & Model

In [ ]:
df = pd.read_csv(CSV_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f'Total rows: {len(df):,}')
print(f'Date range: {df.timestamp.min()} → {df.timestamp.max()}')
print(f'Outage rate: {df.outage.mean():.2%}')
df.head()

In [ ]:
forecaster = OutageForecaster()
forecaster.load(MODEL_PATH)
print('Model loaded.')
print(f'Stored Brier score: {forecaster._brier_score:.4f}')
print(f'Stored MAE duration: {forecaster._mae_duration:.2f} min')

## 2. Held-Out Evaluation (Last 30 Days)

In [ ]:
metrics = evaluate_model(MODEL_PATH, CSV_PATH, holdout_days=30)
print_report(metrics)

## 3. Forecast Uncertainty Band Chart

In [ ]:
# Get last 48 rows for prediction window
window = df.tail(48).reset_index(drop=True)
forecast = forecaster.predict_24h(window)

hours = [f['hour'] for f in forecast]
p_out = [f['p_outage'] for f in forecast]
lower = [f['lower_bound'] for f in forecast]
upper = [f['upper_bound'] for f in forecast]
dur   = [f['duration_min'] for f in forecast]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('24-Hour Outage Forecast — Salon', fontsize=13, fontweight='bold')

# ── P(outage) with uncertainty band ──────────────────────────────────────────
ax1.fill_between(hours, lower, upper, alpha=0.25, color='#ef4444', label='Uncertainty band')
ax1.plot(hours, p_out, color='#ef4444', linewidth=2, marker='o', markersize=4, label='P(outage)')
ax1.axhline(0.30, color='#f59e0b', linestyle='--', linewidth=1, label='Comfort threshold (0.30)')
ax1.axhline(0.15, color='#22c55e', linestyle='--', linewidth=1, label='Luxury threshold (0.15)')
ax1.set_ylabel('P(outage)', fontsize=10)
ax1.set_ylim(0, 1)
ax1.legend(fontsize=8, loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_title('Outage Probability with 80% Confidence Band', fontsize=10)

# ── Expected duration ─────────────────────────────────────────────────────────
colors = ['#22c55e' if p < 0.20 else '#f59e0b' if p < 0.45 else '#ef4444' if p < 0.65 else '#7c3aed' for p in p_out]
ax2.bar(hours, dur, color=colors, alpha=0.8, label='E[duration | outage]')
ax2.set_ylabel('Expected Duration (min)', fontsize=10)
ax2.set_xlabel('Hour of Day', fontsize=10)
ax2.set_xticks(hours)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_title('Expected Outage Duration per Hour', fontsize=10)
patches = [
    mpatches.Patch(color='#22c55e', label='Low (<20%)'),
    mpatches.Patch(color='#f59e0b', label='Medium (20-45%)'),
    mpatches.Patch(color='#ef4444', label='High (45-65%)'),
    mpatches.Patch(color='#7c3aed', label='Critical (>65%)'),
]
ax2.legend(handles=patches, fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('../data/processed/forecast_chart.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 4. Appliance Plan Overlay — Salon

In [ ]:
prioritizer = AppliancePrioritizer(
    appliances_path=BASE_DIR / 'data' / 'raw' / 'appliances.json',
    businesses_path=BASE_DIR / 'data' / 'raw' / 'businesses.json',
)
plan = prioritizer.plan(forecast, 'salon')

print(f"Business: {plan['business']}")
print(f"Date: {plan['date']}")
print(f"Revenue saved: {plan['revenue_saved']:,} RWF")
print(f"Critical hours: {plan['critical_hours']}")

# Display as DataFrame
schedule_df = pd.DataFrame(plan['schedule']).T
schedule_df.index = [f"{h}:00" for h in schedule_df.index]
schedule_df

## 5. Brier Score vs Baseline

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
labels = ['Baseline\n(always mean)', 'Our Model']
scores = [metrics['baseline_brier'], metrics['brier_score']]
colors = ['#94a3b8', '#38bdf8']
bars = ax.bar(labels, scores, color=colors, width=0.4)
ax.axhline(0.10, color='#22c55e', linestyle='--', label='Target < 0.10')
ax.set_ylabel('Brier Score (lower = better)')
ax.set_title('Brier Score Comparison')
ax.legend()
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{score:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, max(scores) * 1.3)
plt.tight_layout()
plt.savefig('../data/processed/brier_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Improvement: {metrics['baseline_comparison']['brier_improvement']}")

## 6. Worst Forecast Hour Analysis

In [ ]:
# Find the hour with highest p_outage in the forecast
worst = max(forecast, key=lambda x: x['p_outage'])
print(f"Worst forecast hour: {worst['hour']}:00")
print(f"  P(outage): {worst['p_outage']:.4f}")
print(f"  Expected duration: {worst['duration_min']:.1f} min")
print(f"  Confidence band: [{worst['lower_bound']:.4f}, {worst['upper_bound']:.4f}]")

# Feature importance for classifier
import pandas as pd
feat_imp = pd.Series(
    forecaster.clf.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)

print('\nTop 10 features by importance:')
print(feat_imp.head(10).to_string())

## 7. Summary Table

In [ ]:
summary = {
    'Metric': ['Brier Score', 'MAE Duration (all)', 'MAE Duration (outages)', 'Lead Time', 'Precision', 'Recall', 'F1'],
    'Value': [
        metrics['brier_score'],
        f"{metrics['mae_duration_all']} min",
        f"{metrics['mae_duration_outage_only']} min",
        f"{metrics['lead_time_minutes']} min",
        metrics['precision'],
        metrics['recall'],
        metrics['f1'],
    ],
    'Target': ['< 0.10', '< 30 min', '< 30 min', '> 60 min', '—', '—', '—'],
    'Status': [
        '✓ PASS' if metrics['targets_met']['brier_lt_010'] else '✗ FAIL',
        '✓ PASS' if metrics['targets_met']['mae_lt_30min'] else '✗ FAIL',
        '—',
        '✓ PASS' if metrics['targets_met']['lead_time_gt_60min'] else '✗ FAIL',
        '—', '—', '—',
    ]
}
pd.DataFrame(summary)